# FOA-Attack laboratory — Kaggle

Notebook này sinh adversarial samples theo **FOA-Attack** và cố ý tách các điểm nghiên cứu thành cell riêng: cấu hình, preprocessing, FOA loss, XAI hook, và attack loop. Baseline mặc định tái lập logic của `jiaxiaojunQAQ/FOA-Attack` (FGSM, K=10) — không thay đổi hyperparameter hay update rule.

**Cách chạy trên Kaggle:** bật GPU + Internet, Add Data `zintom/blackbox-adversarial-attack-dataset`, sau đó chạy tuần tự các cell. Khi smoke test qua, đặt `RUN_ATTACK=True`. File ZIP xuất hiện trong `/kaggle/working/foa_attack_output.zip` ở phần Output của Kaggle.

Điểm mở rộng: đổi `preprocess_image`, `FOAOptimalTransportLoss`, `xai_regularizer`, hoặc `foa_fgsm_attack`. Để báo cáo kết quả FOA paper, luôn giữ `XAI_MODE='none'` và các `PAPER_*` defaults.

## Reproducibility protocol

Revision Git của implementation gốc là **provenance của baseline**, không phải hyperparameter và không được truyền vào hàm attack. Notebook xác minh revision đó trước khi chạy, sau đó lưu lại trong `run_config.json`; điều này cho phép audit và tái lập một kết quả đã báo cáo.

In [ ]:
# Cell 1 — Parameters. Change experimental values here only.
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'FOA-Attack'
OUTPUT_DIR = WORK_ROOT / 'foa_attack_output'
HF_HOME = WORK_ROOT / 'hf_cache'


# Paper / original ensemble_3models.yaml defaults.
PAPER_NUM_SAMPLES = 100
PAPER_STEPS = 300
PAPER_EPSILON = 16.0       # pixel space [0, 255]
PAPER_ALPHA = 1.0          # pixel space [0, 255]
PAPER_INPUT_RES = 224
PAPER_CROP_SCALE = (0.5, 0.9)
PAPER_BACKBONES = ('B16', 'B32', 'Laion')
PAPER_CLUSTER_NUMBER = 10
PAPER_SEED = 2023

# Smoke first; set RUN_ATTACK=True only after it passes.
NUM_SAMPLES = 1
STEPS = 2
RUN_ATTACK = False

# 'none' is parity mode. 'saliency' is an experimental XAI hook, not FOA.
XAI_MODE = 'none'
XAI_WEIGHT = 0.0
DEVICE = 'cuda:0'

assert DATASET_ROOT.exists(), 'Add Kaggle dataset: zintom/blackbox-adversarial-attack-dataset'
assert XAI_MODE in {'none', 'saliency'}
print(f'Dataset: {DATASET_ROOT}\nOutput: {OUTPUT_DIR}\nDevice: {DEVICE}')

In [ ]:
# Cell 2 — Kaggle dependency command. Re-run after changing the Kaggle image.
import sys, subprocess
packages = [
    'transformers==4.49.0', 'tokenizers==0.21.0', 'huggingface-hub==0.28.1',
    'hydra-core==1.3.2', 'omegaconf==2.3.0', 'pytorch-lightning==2.5.0.post0',
    'wandb==0.19.1', 'kmeans-pytorch==0.3', 'ftfy==6.3.1'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages], check=True)
print('Dependencies installed. Restart the Kaggle session if pip reports a Torch/Torchvision conflict.')

In [ ]:
# Cell 3 — Reference implementation protocol (not an attack parameter).
import os, shutil, subprocess, sys
# This immutable record establishes the baseline's provenance. Do not tune it as an experiment setting.
REFERENCE_IMPLEMENTATION = {
    'method': 'FOA-Attack',
    'repository': 'https://github.com/jiaxiaojunQAQ/FOA-Attack.git',
    'revision': '6aa9ad463a1e812b7f77ff2e5c4b1712e7c78c4b',
    'role': 'reference baseline used for implementation-parity verification',
}
os.environ.update({'HF_HOME': str(HF_HOME), 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REFERENCE_IMPLEMENTATION['repository'], str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REFERENCE_IMPLEMENTATION['revision']], check=True)
sys.path.insert(0, str(REPO_DIR))
FOA_SOURCE_REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert FOA_SOURCE_REVISION == REFERENCE_IMPLEMENTATION['revision'], 'Reference source revision mismatch'
print('Verified reference implementation:', REFERENCE_IMPLEMENTATION['method'], FOA_SOURCE_REVISION)

In [ ]:
# Cell 4 — Imports and reproducibility.
import json, random, hashlib, importlib.util
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm.auto import tqdm
from kmeans_pytorch import kmeans

from surrogates import (
    ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor,
    EnsembleFeatureExtractor_ot,
)

def set_foa_seed(seed: int = PAPER_SEED) -> None:
    """Set the same random/CUDA determinism controls as the original FOA script."""
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_foa_seed()
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('Torch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 5 — Dataset discovery and preprocessing (editable research boundary).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(path: Path) -> int:
    """Return recursive number of supported image files below ``path``."""
    return sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))

def find_imagefolder(root: Path, expected_name: str) -> Path:
    """Find the largest ImageFolder named ``expected_name`` in a Kaggle dataset."""
    candidates = sorted((p for p in root.rglob(expected_name) if p.is_dir()), key=lambda p: -count_images(p))
    if not candidates: raise FileNotFoundError(f'Cannot find {expected_name} below {root}')
    return candidates[0]

def pil_to_foa_tensor(image: Image.Image) -> Tensor:
    """Convert PIL RGB image to the original FOA tensor in pixel range [0, 255]."""
    array = np.array(image, np.uint8, copy=True)
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(torch.get_default_dtype())

def preprocess_image(input_res: int = PAPER_INPUT_RES) -> transforms.Compose:
    """Build the exact Resize → CenterCrop → RGB → pixel-tensor preprocessing of FOA."""
    return transforms.Compose([
        transforms.Resize(input_res, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(input_res), transforms.Lambda(lambda x: x.convert('RGB')),
        transforms.Lambda(pil_to_foa_tensor),
    ])

class ImageFolderWithPaths(datasets.ImageFolder):
    """ImageFolder that additionally returns the source image path for manifest export."""
    def __getitem__(self, index: int):
        """Return transformed image, class label, and original filesystem path."""
        image, label = super().__getitem__(index)
        return image, label, self.samples[index][0]

SOURCE_DIR, TARGET_DIR = find_imagefolder(DATASET_ROOT, 'bigscale'), find_imagefolder(DATASET_ROOT, 'target_images')
print('SOURCE_DIR:', SOURCE_DIR, count_images(SOURCE_DIR))
print('TARGET_DIR:', TARGET_DIR, count_images(TARGET_DIR))

In [ ]:
# Cell 6 — FOA surrogate ensemble (unchanged original model choices).
BACKBONE_MAP = {'B16': ClipB16FeatureExtractor, 'B32': ClipB32FeatureExtractor, 'Laion': ClipLaionFeatureExtractor}

def build_foa_ensemble(backbones: Tuple[str, ...] = PAPER_BACKBONES, cluster_number: int = PAPER_CLUSTER_NUMBER):
    """Load original FOA surrogate extractors and its optimal-transport ensemble wrapper."""
    models = [BACKBONE_MAP[name]().eval().to(DEVICE).requires_grad_(False) for name in backbones]
    return EnsembleFeatureExtractor_ot(models, cluster_number=cluster_number), models

# Run only once; it downloads B16, B32 and LAION weights on the first Kaggle run.
ensemble_extractor, surrogate_models = build_foa_ensemble()
print('Loaded:', PAPER_BACKBONES)

In [ ]:
# Cell 7 — FOA loss. This is the paper loss copied into an editable cell.
class FOAOptimalTransportLoss(nn.Module):
    """Original FOA global/local OT loss with its dynamic ensemble weighting (K=10 by default)."""
    def __init__(self, extractors: List[nn.Module], cluster_number: int = PAPER_CLUSTER_NUMBER):
        """Initialize FOA state for the frozen surrogate extractors and local cluster count."""
        super().__init__(); self.extractors = nn.ModuleList(extractors); self.cluster_number = cluster_number
        self.ground_truth, self.ground_truth_local, self.previous_loss_list = [], [], []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor) -> None:
        """Extract target global features and K-means local centers as FOA ground truth."""
        self.ground_truth.clear(); self.ground_truth_local.clear()
        for model in self.extractors:
            global_feature, local_feature = model.global_local_features(x.to(x.device))
            centers = self.cluster_centers(local_feature.squeeze(0), x.device).unsqueeze(0)
            self.ground_truth.append(global_feature); self.ground_truth_local.append(centers)

    def cluster_centers(self, embeddings: Tensor, device: torch.device) -> Tensor:
        """Compute original Euclidean K-means local centers (100 iteration limit)."""
        _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device, iter_limit=100)
        return centers.to(device)

    def optimal_transport(self, source: Tensor, target: Tensor) -> Tensor:
        """Return FOA cosine-similarity optimal-transport score using original Sinkhorn settings."""
        similarity = torch.einsum('md,nd->mn', F.normalize(source, dim=1), F.normalize(target, dim=1)).contiguous()
        kernel = torch.exp(-(1 - similarity) / 0.1)
        u = torch.full((source.shape[0],), 1 / source.shape[0], dtype=similarity.dtype, device=similarity.device)
        v = torch.full((target.shape[0],), 1 / target.shape[0], dtype=similarity.dtype, device=similarity.device)
        with torch.no_grad(): transport = self.sinkhorn(kernel, u, v)
        if torch.isnan(transport).any(): raise FloatingPointError('FOA Sinkhorn returned NaN')
        return torch.sum(transport * similarity)

    def sinkhorn(self, kernel: Tensor, u: Tensor, v: Tensor) -> Tensor:
        """Run the original 100-step, 1e-2-threshold Sinkhorn normalization."""
        r, c = torch.ones_like(u), torch.ones_like(v)
        for _ in range(100):
            old_r = r; r = u / (kernel @ c.unsqueeze(-1)).squeeze(-1); c = v / (kernel.T @ r.unsqueeze(-1)).squeeze(-1)
            if (r - old_r).abs().mean().item() < 1e-2: break
        return torch.outer(r, c) * kernel

    def forward(self, global_features: Dict[int, Tensor], local_features: Dict[int, Tensor]) -> Tensor:
        """Compute original FOA global OT + 0.2 local OT and dynamic model weights."""
        losses = [self.optimal_transport(self.ground_truth[i], global_features[i].unsqueeze(0)) + 0.2 * self.optimal_transport(self.ground_truth_local[i].squeeze(0), local_features[i].squeeze(0)) for i in range(len(self.extractors))]
        if not self.previous_loss_list: self.previous_loss_list = [loss.detach() for loss in losses]
        weights = np.array([loss.item() / (prev.item() + 1e-8) for loss, prev in zip(losses, self.previous_loss_list)])
        weights = np.exp(weights) / np.exp(weights).sum() * len(weights)
        self.previous_loss_list = [loss.detach() for loss in losses]
        return sum(float(weights[i]) * loss for i, loss in enumerate(losses))

foa_loss = FOAOptimalTransportLoss(surrogate_models)

In [ ]:
# Cell 8 — Optional XAI boundary. 'none' returns exactly zero and preserves FOA baseline.
def xai_regularizer(adv_image: Tensor, foa_score: Tensor, mode: str = XAI_MODE, weight: float = XAI_WEIGHT) -> Tensor:
    """Return an optional XAI research term; use ``none``/0 for paper-parity FOA.

    ``saliency`` penalizes diffuse input gradients. It is deliberately opt-in and changes the method.
    """
    if mode == 'none' or weight == 0.0: return foa_score.new_zeros(())
    if mode == 'saliency':
        saliency = torch.autograd.grad(foa_score, adv_image, retain_graph=True, create_graph=True)[0].abs()
        return weight * saliency.flatten(1).std(dim=1).mean()
    raise ValueError(f'Unknown XAI mode: {mode}')

print('XAI mode:', XAI_MODE, '| weight:', XAI_WEIGHT)

In [ ]:
# Cell 9 — FGSM FOA optimization loop (editable research boundary).
def foa_fgsm_attack(image_org: Tensor, image_tgt: Tensor, extractor: nn.Module, loss_fn: nn.Module, source_crop: nn.Module, target_crop: nn.Module, steps: int = STEPS, epsilon: float = PAPER_EPSILON, alpha: float = PAPER_ALPHA) -> Tensor:
    """Generate a targeted FOA adversarial image using the original iterative FGSM update.

    Input tensors are in [0,255]; returned tensor is clamped to [0,1], exactly as the source script.
    With XAI_MODE='none', loss selection, gradient sign update, projection, and target crops match FOA.
    """
    delta = torch.zeros_like(image_org, requires_grad=True)
    for _ in tqdm(range(steps), desc='FOA attack'):
        with torch.no_grad(): loss_fn.set_ground_truth(target_crop(image_tgt))
        adv_image = image_org + delta
        global_features, local_features = extractor(adv_image)
        global_score = loss_fn(global_features, local_features)
        if isinstance(source_crop, nn.Identity):
            objective = global_score
        else:
            crop_global, crop_local = extractor(source_crop(adv_image))
            objective = loss_fn(crop_global, crop_local)
        objective = objective + xai_regularizer(adv_image, objective)
        gradient = torch.autograd.grad(objective, delta, create_graph=False)[0]
        delta.data = torch.clamp(delta + alpha * torch.sign(gradient), min=-epsilon, max=epsilon)
    return torch.clamp((image_org + delta) / 255.0, 0.0, 1.0)

SOURCE_CROP = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)
TARGET_CROP = transforms.RandomResizedCrop(PAPER_INPUT_RES, scale=PAPER_CROP_SCALE)

In [ ]:
# Cell 10 — Parity guard: validate notebook update against original source on a deterministic mock.
def assert_paper_defaults() -> None:
    """Fail early if a paper-parity run has changed an original FOA default."""
    assert (PAPER_EPSILON, PAPER_ALPHA, PAPER_STEPS, PAPER_CLUSTER_NUMBER) == (16.0, 1.0, 300, 10)
    assert PAPER_BACKBONES == ('B16', 'B32', 'Laion') and PAPER_CROP_SCALE == (0.5, 0.9)

def verify_update_parity() -> None:
    """Compare original and notebook FGSM tensor updates with the same mock feature/loss contract."""
    assert_paper_defaults()
    import ast
    source = (REPO_DIR / 'generate_adversarial_samples_foa_attack.py').read_text(encoding='utf-8')
    assert 'delta + cfg.optim.alpha * torch.sign(grad)' in source and 'min=-cfg.optim.epsilon' in source
    class MockExtractor(nn.Module):
        """Minimal differentiable feature extractor for parity testing only."""
        def forward(self, x):
            """Return mean global/local features matching the FOA extractor contract."""
            return {0: x.mean((2, 3)).squeeze(0)}, {0: x.mean((2, 3)).unsqueeze(1)}
    class MockLoss(nn.Module):
        """Target-feature similarity loss for the deterministic parity test only."""
        def set_ground_truth(self, x):
            """Cache mean target features."""
            self.target = x.mean((2, 3)).detach()
        def forward(self, global_f, local_f):
            """Return a differentiable score whose ascent moves input toward target."""
            return -(global_f[0] - self.target.squeeze(0)).square().mean()
    x, y = torch.full((1, 3, 4, 4), 100.0, device=DEVICE), torch.full((1, 3, 4, 4), 130.0, device=DEVICE)
    ours = foa_fgsm_attack(x, y, MockExtractor(), MockLoss(), nn.Identity(), nn.Identity(), steps=3, epsilon=16, alpha=1)
    tree = ast.parse(source); original_node = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'fgsm_attack')
    namespace = {'torch': torch, 'nn': nn, 'tqdm': lambda sequence, **_: sequence, 'log_metrics': lambda *_, **__: None, 'MainConfig': object, 'Optional': Optional, 'transforms': transforms}
    exec(compile(ast.Module(body=[original_node], type_ignores=[]), '<pinned-foa>', 'exec'), namespace)
    from types import SimpleNamespace
    cfg = SimpleNamespace(optim=SimpleNamespace(steps=3, epsilon=16, alpha=1), model=SimpleNamespace(use_source_crop=False))
    original = namespace['fgsm_attack'](cfg, MockExtractor(), MockLoss(), nn.Identity(), nn.Identity(), 0, x, y)
    assert torch.allclose(ours, original), 'Notebook FGSM loop differs from pinned FOA source'
    print('Parity guard passed: notebook output equals pinned original FGSM on the deterministic mock.')

verify_update_parity()

In [ ]:
# Cell 11 — Run attack and write Kaggle-ready PNGs + manifest.
def run_foa_attack() -> Path:
    """Attack paired ImageFolder samples and write source, target, adversarial PNGs and manifest.jsonl."""
    if XAI_MODE == 'none': assert_paper_defaults()
    if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
    (OUTPUT_DIR / 'images').mkdir(parents=True)
    source_ds = ImageFolderWithPaths(SOURCE_DIR, transform=preprocess_image())
    target_ds = ImageFolderWithPaths(TARGET_DIR, transform=preprocess_image())
    records = []
    for index, ((source, _, source_path), (target, _, target_path)) in enumerate(zip(DataLoader(source_ds, batch_size=1, shuffle=False), DataLoader(target_ds, batch_size=1, shuffle=False))):
        if index >= NUM_SAMPLES: break
        source, target = source.to(DEVICE), target.to(DEVICE)
        adversarial = foa_fgsm_attack(source, target, ensemble_extractor, foa_loss, SOURCE_CROP, TARGET_CROP)
        stem = f'{index:05d}'
        paths = {'source': f'images/{stem}_source.png', 'target': f'images/{stem}_target.png', 'adversarial': f'images/{stem}_adversarial.png'}
        save_image(source / 255.0, OUTPUT_DIR / paths['source']); save_image(target / 255.0, OUTPUT_DIR / paths['target']); save_image(adversarial, OUTPUT_DIR / paths['adversarial'])
        saved = np.asarray(Image.open(OUTPUT_DIR / paths['adversarial'])).astype(np.int16)
        original = np.asarray(Image.open(OUTPUT_DIR / paths['source'])).astype(np.int16)
        linf = int(np.abs(saved - original).max())
        if linf > int(PAPER_EPSILON): raise RuntimeError(f'PNG violates L∞ bound: {linf}')
        records.append({'index': index, 'original_path': source_path[0], 'target_path': target_path[0], 'linf_255': linf, **paths})
    with (OUTPUT_DIR / 'manifest.jsonl').open('w', encoding='utf-8') as handle:
        handle.write('\n'.join(json.dumps(row) for row in records) + ('\n' if records else ''))
    (OUTPUT_DIR / 'run_config.json').write_text(json.dumps({'reference_implementation': REFERENCE_IMPLEMENTATION, 'resolved_revision': FOA_SOURCE_REVISION, 'num_samples': NUM_SAMPLES, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA, 'xai_mode': XAI_MODE}, indent=2), encoding='utf-8')
    return OUTPUT_DIR

if RUN_ATTACK:
    print('Finished:', run_foa_attack())
else:
    print('Dry run only. Set RUN_ATTACK=True in Cell 1, then re-run Cells 9 and 11.')

In [ ]:
# Cell 12 — Preview and ZIP for Kaggle Output/download.
import matplotlib.pyplot as plt

def export_kaggle_zip(output_dir: Path = OUTPUT_DIR) -> Path:
    """Create a ZIP archive that Kaggle exposes under /kaggle/working Output."""
    if not (output_dir / 'manifest.jsonl').exists(): raise FileNotFoundError('Run Cell 11 first.')
    return Path(shutil.make_archive(str(WORK_ROOT / 'foa_attack_output'), 'zip', root_dir=output_dir))

if (OUTPUT_DIR / 'manifest.jsonl').exists():
    record = json.loads((OUTPUT_DIR / 'manifest.jsonl').read_text().splitlines()[0])
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for axis, role in zip(axes, ('source', 'target', 'adversarial')):
        axis.imshow(Image.open(OUTPUT_DIR / record[role])); axis.set_title(role); axis.axis('off')
    plt.show()
    print('ZIP:', export_kaggle_zip())
else:
    print('No output yet. Run Cell 11 with RUN_ATTACK=True.')